# 🎵 Multi-Agent AI Music Producer

Generate full audio tracks segment-by-segment using a multi-agent system orchestrated with LangGraph.

**Features:**
- Reference track analysis for style matching
- Intelligent track planning with segment prompts
- MusicGen-powered audio generation
- Quality control via critic feedback loop
- Professional mastering with normalization

---

## 1. Environment Setup

First, let's install dependencies and clone the repository.

In [ ]:
# Check runtime type
import torch

if torch.cuda.is_available():
    print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected. Generation will be slow.")
    print("   Go to Runtime > Change runtime type > GPU")

In [ ]:
# Install dependencies
!pip install -q langgraph>=0.2.0 pydantic>=2.0.0 pyyaml
!pip install -q anthropic openai huggingface_hub transformers
!pip install -q librosa>=0.10.0 soundfile scipy numpy
!pip install -q audiocraft>=1.3.0  # This may take a few minutes

In [ ]:
# Clone the repository (if running from Colab without the local files)
import os

if not os.path.exists('multi_agent_ai_music_producer'):
    # Replace with your repository URL
    !git clone https://github.com/YOUR_USERNAME/multi_agent_ai_music_producer.git
    %cd multi_agent_ai_music_producer
else:
    %cd multi_agent_ai_music_producer
    !git pull

In [ ]:
# Add repository to path
import sys
sys.path.insert(0, os.getcwd())

## 2. Configuration

Set your API keys and configure the generation parameters.

In [ ]:
# Set API keys (use Colab secrets or enter manually)
from google.colab import userdata

try:
    # Try to get from Colab secrets
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print("✅ Loaded ANTHROPIC_API_KEY from Colab secrets")
except:
    # Manual entry
    import getpass
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')

# Optional: OpenAI key for GPT models
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
# Install HuggingFace dependencies
!pip install -q transformers accelerate bitsandbytes

# Optional: Login to HuggingFace for gated models (Llama)
from huggingface_hub import login
login()  # Enter your HF token when prompted

In [ ]:
from src.config import Settings

# Create configuration
settings = Settings(
    llm={
        'provider': 'huggingface',
        'model': 'mistralai/Mistral-7B-Instruct-v0.3',  # Free, no API key needed
        'temperature': 0.7,
        'max_tokens': 4096,
    },
    generation={
        'segment_duration_sec': 15.0,      # Length of each segment (max 30s for MusicGen)
        'max_retries': 3,                   # Retries per segment if critic rejects
        'approval_threshold': 0.7,          # Minimum score for approval
    },
    audio={
        'sample_rate': 32000,               # MusicGen native rate
        'output_dir': 'output',
        'target_lufs': -14.0,               # Streaming loudness standard
    },
    logging={
        'level': 'standard',
        'log_dir': 'logs',
        'enable_json': True,
    },
)

print("✅ Configuration loaded")
print(f"   LLM: {settings.llm.provider}/{settings.llm.model}")
print(f"   Segment duration: {settings.generation.segment_duration_sec}s")

## 3. Upload Reference Tracks (Optional)

Upload reference audio files for style analysis. The system will extract BPM, key, energy, and instrument characteristics.

In [ ]:
from google.colab import files
from pathlib import Path

# Create references directory
ref_dir = Path('references')
ref_dir.mkdir(exist_ok=True)

# Upload files
print("Upload reference tracks (WAV, MP3, or FLAC):")
uploaded = files.upload()

# Save uploaded files
reference_paths = []
for filename, content in uploaded.items():
    filepath = ref_dir / filename
    with open(filepath, 'wb') as f:
        f.write(content)
    reference_paths.append(str(filepath))
    print(f"  ✅ Saved: {filename}")

print(f"\n{len(reference_paths)} reference file(s) uploaded")

In [ ]:
# Or skip upload and use no references
# reference_paths = []

## 4. Define Your Prompt

Describe the music you want to generate. Be specific about:
- Genre and style
- Mood and energy
- Instruments and sounds
- Structure (intro, verse, chorus, etc.)

In [ ]:
# Your creative prompt
user_prompt = """
Create an ambient electronic track with the following characteristics:
- Starts with soft, evolving synth pads and gentle textures
- Gradually builds with subtle rhythmic elements around the midpoint
- Incorporates atmospheric sounds like distant bells and reverbed piano notes
- Maintains a dreamy, meditative quality throughout
- Ends with a gradual fade back to soft pads
"""

# Target duration in seconds
target_duration_sec = 90.0  # 1.5 minutes

print(f"Prompt: {user_prompt[:100]}...")
print(f"Target duration: {target_duration_sec}s ({target_duration_sec/60:.1f} minutes)")

## 5. Run the Music Producer

Execute the multi-agent workflow to generate your track.

In [ ]:
from src import run_workflow
from src.logging.progress import ProgressCallback
from IPython.display import display, clear_output
import ipywidgets as widgets

# Progress display
output_widget = widgets.Output()
progress_bar = widgets.FloatProgress(
    value=0.0, min=0.0, max=1.0,
    description='Progress:',
    bar_style='info',
    style={'bar_color': '#3498db'},
    layout=widgets.Layout(width='100%')
)
status_label = widgets.HTML(value='<b>Initializing...</b>')

display(widgets.VBox([status_label, progress_bar, output_widget]))

# Custom progress callback for Colab
class ColabProgressCallback(ProgressCallback):
    def on_workflow_start(self, state):
        status_label.value = '<b>🚀 Starting workflow...</b>'
    
    def on_agent_start(self, agent_name, state):
        status_label.value = f'<b>🤖 Agent: {agent_name}</b>'
        with output_widget:
            clear_output(wait=True)
            print(f"Agent '{agent_name}' working...")
    
    def on_agent_complete(self, agent_name, state):
        with output_widget:
            print(f"✅ Agent '{agent_name}' completed")
    
    def on_segment_start(self, segment_index, total_segments, state):
        progress_bar.value = segment_index / total_segments
        status_label.value = f'<b>🎵 Generating segment {segment_index + 1}/{total_segments}</b>'
    
    def on_segment_complete(self, segment_index, total_segments, state):
        progress_bar.value = (segment_index + 1) / total_segments
    
    def on_workflow_complete(self, state):
        progress_bar.value = 1.0
        progress_bar.bar_style = 'success'
        status_label.value = '<b>✅ Track generation complete!</b>'

progress = ColabProgressCallback()

In [ ]:
# Run the workflow
try:
    final_state = run_workflow(
        user_prompt=user_prompt,
        reference_paths=reference_paths,
        target_duration_sec=target_duration_sec,
        settings=settings,
        progress_callback=progress,
    )
    
    print("\n" + "="*50)
    print("🎉 GENERATION COMPLETE!")
    print("="*50)
    print(f"Output: {final_state.get('final_output_path', 'N/A')}")
    print(f"Status: {final_state.get('status', 'unknown')}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    raise

## 6. Review Results

Listen to the generated track and review the generation details.

In [ ]:
# Display audio player
from IPython.display import Audio, display

output_path = final_state.get('final_output_path')

if output_path and os.path.exists(output_path):
    print(f"🎧 Playing: {output_path}")
    display(Audio(filename=output_path))
else:
    print("❌ No output file found")

In [ ]:
# View musical profile
import json

profile = final_state.get('musical_profile')
if profile:
    print("🎼 Musical Profile:")
    print(f"   BPM: {profile.get('bpm', 'N/A')}")
    print(f"   Key: {profile.get('key', 'N/A')} {profile.get('mode', '')}")
    print(f"   Energy: {profile.get('energy_target', 'N/A')}")
    print(f"   Style: {', '.join(profile.get('style_descriptors', []))}")
    print(f"   Instruments: {', '.join(profile.get('instrument_suggestions', []))}")

In [ ]:
# View track plan
plan = final_state.get('track_plan')
if plan:
    print("📋 Track Plan:")
    print(f"   Total duration: {plan.get('total_duration_sec', 0):.1f}s")
    print(f"   Segments: {plan.get('segment_count', 0)}")
    print("\n   Segment breakdown:")
    for i, (dur, prompt) in enumerate(zip(
        plan.get('segment_durations', []),
        plan.get('segment_prompts', [])
    )):
        print(f"   {i+1}. [{dur:.1f}s] {prompt[:60]}...")

In [ ]:
# View segment details
segments = final_state.get('completed_segments', [])
if segments:
    print(f"\n🔊 Generated Segments ({len(segments)}):")
    for seg in segments:
        fb = seg.get('critic_feedback', {})
        status = '✅' if fb.get('approved') else '⚠️'
        scores = f"Q:{fb.get('quality_score', 0):.2f} C:{fb.get('consistency_score', 0):.2f}"
        print(f"   {status} Segment {seg.get('segment_index', '?')}: {seg.get('duration_sec', 0):.1f}s - {scores}")

## 7. Download Output

Download the generated track to your computer.

In [ ]:
# Download final output
from google.colab import files

if output_path and os.path.exists(output_path):
    print(f"📥 Downloading: {output_path}")
    files.download(output_path)
else:
    print("No output file to download")

In [ ]:
# Download individual segments (optional)
download_segments = False  # Set to True to download all segments

if download_segments:
    from pathlib import Path
    segments = final_state.get('completed_segments', [])
    for seg in segments:
        seg_path = seg.get('audio_path')
        if seg_path and os.path.exists(seg_path):
            print(f"📥 Downloading segment {seg.get('segment_index')}")
            files.download(seg_path)

## 8. Advanced: View Workflow Visualization

See the LangGraph workflow structure.

In [ ]:
from src.graph.workflow import MusicProducerGraph, get_workflow_visualization
from IPython.display import display, Markdown

# Create and visualize the workflow
graph = MusicProducerGraph(settings=settings)
graph.build()

mermaid_diagram = get_workflow_visualization(graph)

# Display as Mermaid (if supported)
display(Markdown(f"```mermaid\n{mermaid_diagram}\n```"))

# Or just print it
print(mermaid_diagram)

## 9. Cleanup

Free up GPU memory and clean temporary files.

In [ ]:
# Clear GPU cache
import torch
import gc

# Clear MusicGen from memory
try:
    from src.tools.audio_generation import MusicGenWrapper
    MusicGenWrapper._instance = None
except:
    pass

gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    print(f"GPU memory freed. Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Optional: Clean up temporary files
cleanup_temp = False  # Set to True to clean up

if cleanup_temp:
    import shutil
    
    # Remove output directory (keep final track)
    output_dir = Path('output')
    if output_dir.exists():
        # Keep final track, remove segments
        for f in output_dir.glob('segment_*.wav'):
            f.unlink()
        print("Cleaned up segment files")
    
    # Remove logs
    log_dir = Path('logs')
    if log_dir.exists():
        shutil.rmtree(log_dir)
        print("Cleaned up logs")

---

## Tips & Troubleshooting

### GPU Memory Issues
- Use shorter `segment_duration_sec` (10-15s works well)
- Run the cleanup cell to free memory between generations
- If OOM errors persist, restart the runtime

### Quality Improvements
- Upload reference tracks for better style matching
- Be specific in your prompt about instruments and mood
- Increase `max_retries` for more quality iterations
- Lower `approval_threshold` if critic is too strict

### API Costs
- Use Ollama with local models for free LLM inference
- Reduce `max_retries` to minimize LLM calls
- Consider HuggingFace Inference API for lower costs

### Common Errors
- **"No GPU detected"**: Change runtime type to GPU
- **"API key not found"**: Set your API key in the configuration cell
- **"audiocraft not installed"**: Run the dependency cell and wait for completion